# AI Student Impact Exploratory Data Analysis

This notebook profiles the locally provisioned, ignored AI Student Impact CSV. Set `AI_STUDENT_IMPACT_CSV` to an explicit CSV path, or run the notebook from the repository root to use the documented default `data/01-raw/ai-student-impact/ai_student_impact_dataset.csv`. The notebook never downloads data, trains or loads a model, or writes analytical outputs.

The records are synthetic portfolio evidence. Every result below is descriptive and non-causal; it does not establish effects of AI use, clinical burnout risk, measurement validity, or generalization to real students.

In [ ]:
import hashlib
import os
from pathlib import Path

import pandas as pd

CSV_ENVIRONMENT_VARIABLE = "AI_STUDENT_IMPACT_CSV"
DEFAULT_CSV_PATH = Path("data/01-raw/ai-student-impact/ai_student_impact_dataset.csv")
csv_path = Path(os.environ.get(CSV_ENVIRONMENT_VARIABLE, DEFAULT_CSV_PATH)).expanduser()
if not csv_path.is_file():
    raise FileNotFoundError(
        f"AI Student Impact CSV not found at {csv_path}. "
        f"Set {CSV_ENVIRONMENT_VARIABLE} to the ignored local CSV path."
    )

dataframe = pd.read_csv(csv_path)
EXPECTED_COLUMNS = (
    "Student_ID",
    "Major_Category",
    "Year_of_Study",
    "Pre_Semester_GPA",
    "Weekly_GenAI_Hours",
    "Primary_Use_Case",
    "Prompt_Engineering_Skill",
    "Tool_Diversity",
    "Paid_Subscription",
    "Traditional_Study_Hours",
    "Perceived_AI_Dependency",
    "Institutional_Policy",
    "Anxiety_Level_During_Exams",
    "Post_Semester_GPA",
    "Skill_Retention_Score",
    "Burnout_Risk_Level",
)
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(dataframe.columns))
unexpected_columns = sorted(set(dataframe.columns) - set(EXPECTED_COLUMNS))
if missing_columns or unexpected_columns:
    raise ValueError(
        "Expected the 16-column AI Student Impact schema; "
        f"missing={missing_columns}, unexpected={unexpected_columns}"
    )

## 1. Dataset shape, identity, schema, and data quality

The SHA-256 digest identifies the exact local bytes used for this run. It is not a license, provenance, or validity guarantee. `Student_ID` is treated only as a key.

In [ ]:
dataset_shape = dataframe.shape
dataset_identity = {
    "file_name": csv_path.name,
    "sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    "rows": dataset_shape[0],
    "columns": dataset_shape[1],
}
schema_summary = dataframe.dtypes.astype(str).rename("dtype").to_frame()
schema_summary["non_null"] = dataframe.notna().sum()
schema_summary["unique"] = dataframe.nunique(dropna=True)
missing_values = dataframe.isna().sum().rename("missing_count").to_frame()
missing_values["missing_percent"] = (
    missing_values["missing_count"] / len(dataframe) * 100
)
duplicate_row_count = int(dataframe.duplicated().sum())
duplicate_student_id_count = int(dataframe["Student_ID"].duplicated().sum())

## 2. Target distribution

`Burnout_Risk_Level` is a synthetic generated label, not a diagnosis or validated psychological instrument. Counts and proportions describe only this file.

In [ ]:
target_distribution = (
    dataframe["Burnout_Risk_Level"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)
target_distribution["proportion"] = (
    target_distribution["count"] / len(dataframe)
)

## 3. Numeric and categorical feature summaries

These summaries include separate post-outcome fields for description. They do not authorize those fields as burnout predictors.

In [ ]:
IDENTIFIER_COLUMN = "Student_ID"
numeric_columns = [
    column
    for column in dataframe.select_dtypes(include="number").columns
    if column != IDENTIFIER_COLUMN
]
categorical_columns = [
    column
    for column in dataframe.columns
    if column not in numeric_columns and column != IDENTIFIER_COLUMN
]
numeric_summary = dataframe[numeric_columns].describe().T
categorical_summary = dataframe[categorical_columns].describe().T
feature_value_counts = {
    column: dataframe[column].value_counts(dropna=False)
    for column in categorical_columns
    if column not in {"Student_ID", "Burnout_Risk_Level"}
}

## 4. Five academic categories

The research scope names Arts, Humanities, Medical, Business, and STEM. Support and synthetic target distributions are shown for those categories where present; zeros make absent categories explicit rather than silently dropping them.

In [ ]:
ACADEMIC_CATEGORIES = ("Arts", "Humanities", "Medical", "Business", "STEM")
academic_major = pd.Categorical(
    dataframe["Major_Category"], categories=ACADEMIC_CATEGORIES
)
academic_category_summary = (
    pd.Series(academic_major, name="Major_Category")
    .value_counts(sort=False)
    .rename("count")
    .to_frame()
)
academic_category_summary["proportion"] = (
    academic_category_summary["count"] / len(dataframe)
)
academic_target_distribution = pd.crosstab(
    academic_major, dataframe["Burnout_Risk_Level"], dropna=False
)

## 5. Descriptive pre/post GPA difference

The synthetic pre/post pair is not a dated longitudinal follow-up or an intervention estimate. The difference below is descriptive only and MUST NOT be read as an effect of AI use.

In [ ]:
gpa_difference = (
    dataframe["Post_Semester_GPA"] - dataframe["Pre_Semester_GPA"]
).rename("Post_minus_Pre_GPA")
gpa_difference_summary = gpa_difference.describe()
gpa_difference_by_academic_category = (
    gpa_difference.groupby(academic_major, observed=False)
    .agg(["count", "mean", "median", "std"])
)

## 6. Correlations and categorical associations

Spearman correlations and normalized cross-tabs are exploratory associations, not causal effects. They may reflect synthetic generator rules, unknown label construction, shared inputs, or post-outcome timing. No p-values, causal adjustment, clinical interpretation, or external-student inference is claimed.

In [ ]:
numeric_correlations = dataframe[numeric_columns].corr(method="spearman")
high_burnout_indicator = dataframe["Burnout_Risk_Level"].eq("High").astype(int)
target_numeric_associations = (
    dataframe[numeric_columns]
    .corrwith(high_burnout_indicator, method="spearman")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
target_associations = {
    column: pd.crosstab(
        dataframe[column], dataframe["Burnout_Risk_Level"], normalize="index"
    )
    for column in categorical_columns
    if column not in {"Student_ID", "Burnout_Risk_Level"}
}

## Interpretation boundary

Use these tables to inspect this exact synthetic file and formulate questions, not to rank real students or recommend interventions. Any real-student study requires validated constructs, ethical review, prespecified methods, uncertainty analysis, and genuinely external evidence.